# LSTM + Random Forest Hybrid (Hybrid 1) — CORRECTED PIPELINE

**This notebook supersedes the previous Colab-based run.**

Change log (2026-04-24):
- Loads the post-alignment-fix CSVs `fraudTrain_engineered_with_ids.csv` / `fraudTest_engineered_with_ids.csv` (composite-key merge, validated one-to-one — see `AUDIT_WIN_NARRATIVE.md`).
- Removes the broken positional `.values` reattachment of `cc_num` from the raw CSV.
- Removes Google Drive mount / Colab paths.
- Adds fixed random seeds for reproducibility.

**Corrected LSTM+RF F1 = 0.7892** (from `run_gap_experiments.py`, seed=42, recorded in `verified_metrics.json["gap_experiments"]["LSTM_reproduced_baseline"]`).

The historical F1 = 0.4747 from the original Colab run was produced by the broken positional-attach pipeline and is preserved only in `FYP_Hybrid_Model_BROKEN.ipynb` for audit. It should not be cited as the current Hybrid 1 result.


In [1]:
# Reproducibility seeds
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except ImportError:
    pass

try:
    import tensorflow as tf
    tf.random.set_seed(SEED)
except ImportError:
    pass


In [2]:
import pandas as pd
import numpy as np

# Post-fix CSVs: cc_num and trans_date_trans_time come from a validated
# composite-key one-to-one merge (see AUDIT_WIN_NARRATIVE.md).
df_train = pd.read_csv('../data/engineered/fraudTrain_engineered_with_ids.csv')
df_test  = pd.read_csv('../data/engineered/fraudTest_engineered_with_ids.csv')

target = 'is_fraud'
drop_cols = ['is_fraud', 'unix_time']  # features exclude these; cc_num/trans_date_trans_time
                                        # stay in X_train/X_test for sequence grouping only

X_train = df_train.drop(columns=drop_cols)
y_train = df_train[target]
X_test  = df_test.drop(columns=drop_cols)
y_test  = df_test[target]

print(f"Train: {X_train.shape} | Fraud: {y_train.sum()}")
print(f"Test:  {X_test.shape} | Fraud: {y_test.sum()}")
print(f"Unique cards in train: {X_train['cc_num'].nunique()}")
print(f"Unique cards in test:  {X_test['cc_num'].nunique()}")


Train: (1296675, 16) | Fraud: 7506
Test:  (555719, 16) | Fraud: 2145
Unique cards in train: 983
Unique cards in test:  924


## Note on cardholder IDs

The original notebook attached `cc_num` and `trans_date_trans_time` by positional `.values` from the raw CSV, assuming row `i` of the engineered file corresponded to row `i` of the raw file. It did not — the EDA notebook sorts rows by `(cc_num, trans_datetime)` during velocity computation, so positional alignment was broken (0 / 1,296,675 rows matched — see `verify_alignment_bug.py`).

In this corrected notebook, `cc_num` and `trans_date_trans_time` are already present in the engineered CSV via a validated `(unix_time, amt, city_pop)` composite-key merge (`fix_lstm_alignment.py`, `validate="one_to_one"`, plus a 20-row round-trip sanity check). The positional reattachment step is therefore removed entirely.


# LSTM + Random Forest Hybrid Model

## Data Loading and Sequence Preparation

This notebook builds the hybrid LSTM + Random Forest model for credit card fraud detection. The same pre-processed engineered datasets from the EDA notebook are loaded, containing 1,296,675 training transactions and 555,719 test transactions across 14 features.

The key difference from the XGBoost standalone notebook is the addition of transaction sequencing. XGBoost treats every transaction independently — it reads one row at a time with no knowledge of what happened before. The LSTM component of the hybrid model requires the opposite: it reads a sequence of recent transactions from the same card in chronological order, looking for patterns in how spending behaviour changes over time.

To build these sequences, the original card numbers and timestamps are temporarily re-attached to the engineered features. Card numbers are used to group transactions by cardholder, and timestamps ensure the sequences are in the correct chronological order. These columns are used only for grouping and sorting — they are not fed into the model as features.

In [3]:
# Architecture diagram
from IPython.display import Image, display
import os

img_path = "hybrid_architecture.png"
if os.path.exists(img_path):
    display(Image(img_path, width=1000))
else:
    print("LSTM + Random Forest Hybrid Architecture:")
    print("  Input (14 features) -> Sequence Builder (5 timesteps) -> LSTM (64 units)")
    print("  -> Dropout (0.3) -> Dense (32) -> Dense (16) -> Fraud Probability")
    print("  Fraud Probability + 14 Static Features -> Random Forest -> Final Prediction")


LSTM + Random Forest Hybrid Architecture:
  Input (14 features) -> Sequence Builder (5 timesteps) -> LSTM (64 units)
  -> Dropout (0.3) -> Dense (32) -> Dense (16) -> Fraud Probability
  Fraud Probability + 14 Static Features -> Random Forest -> Final Prediction


In [4]:
# ============================================================
# CELL 4: BUILD TRANSACTION SEQUENCES FOR LSTM
# ============================================================
# Why sequences? XGBoost reads one transaction at a time.
# LSTM reads multiple transactions in ORDER — like reading
# a card's recent history. If a card had 5 purchases in
# the last hour, LSTM sees all 5 in order and spots the
# pattern. That's the whole point of the hybrid approach.
# ============================================================

from sklearn.preprocessing import StandardScaler
import numpy as np

# ----------------------------------------------------------
# Step 1: Identify which columns are actual features
# We added cc_num and trans_date_trans_time temporarily
# just for grouping and sorting. They are NOT model features.
# ----------------------------------------------------------
feature_cols = [c for c in X_train.columns if c not in ['cc_num', 'trans_date_trans_time']]
print(f"Features for model: {feature_cols}")
print(f"Number of features: {len(feature_cols)}")

# ----------------------------------------------------------
# Step 2: Scale all features to same range
# Why? LSTM is a neural network. Neural networks struggle
# when features have wildly different scales.
# amt ranges 0-10000, city_pop ranges 0-2000000, hour 0-23.
# StandardScaler converts everything to mean=0, std=1.
# fit_transform on train — learns the scaling rules
# transform on test — applies same rules (no data leakage)
# ----------------------------------------------------------
scaler = StandardScaler()
X_train[feature_cols] = scaler.fit_transform(X_train[feature_cols])
X_test[feature_cols] = scaler.transform(X_test[feature_cols])

# ----------------------------------------------------------
# Step 3: Sort by card number and time
# Why? LSTM reads transactions in chronological order.
# We need card 4521's transactions sorted oldest → newest.
# Without sorting, the sequences would be random order
# and LSTM can't learn any temporal patterns.
# We also temporarily attach the labels (y) so they stay
# aligned after sorting, then detach them again.
# ----------------------------------------------------------
X_train['y'] = y_train.values
X_train = X_train.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)
y_train = X_train['y']
X_train = X_train.drop(columns=['y'])

X_test['y'] = y_test.values
X_test = X_test.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)
y_test = X_test['y']
X_test = X_test.drop(columns=['y'])

# ----------------------------------------------------------
# Step 4: Define sequence length
# SEQ_LEN = 5 means for each transaction, LSTM looks at
# the LAST 5 transactions on that same card (including
# the current one). Like a sliding window.
#
# Example: Card 4521 has transactions T1, T2, T3, T4, T5, T6
# For T5, the sequence is [T1, T2, T3, T4, T5]
# For T6, the sequence is [T2, T3, T4, T5, T6]
#
# Why 5? Short enough to run fast on Colab.
# Long enough to capture rapid fraud patterns.
# ----------------------------------------------------------
SEQ_LEN = 5

# ----------------------------------------------------------
# Step 5: Function to build sequences per card
#
# For each transaction, this function:
# 1. Looks at the same card's previous transactions
# 2. Grabs the last SEQ_LEN transactions (the sequence)
# 3. If the card has fewer than SEQ_LEN transactions so far,
#    pads with zeros (e.g. card's 2nd ever transaction
#    gets [0, 0, 0, T1, T2] as its sequence)
# 4. Also saves the current transaction's static features
#    (these go to Random Forest, not LSTM)
# 5. Saves the label (fraud or legit) for that transaction
#
# Returns three arrays:
# - sequences: shape (num_transactions, SEQ_LEN, num_features)
#   This is what LSTM reads — 3D array
# - static_features: shape (num_transactions, num_features)
#   This is what Random Forest reads — 2D array
# - targets: shape (num_transactions,)
#   The fraud/legit labels
# ----------------------------------------------------------
def build_sequences(df, labels, feature_cols, seq_len):
    sequences = []        # Will hold all LSTM sequences
    static_features = []  # Will hold all RF static features
    targets = []          # Will hold all labels

    # Group all transactions by card number
    # Each group = one card's full transaction history
    grouped = df.groupby('cc_num')

    for card, group in grouped:
        # Get the feature values for this card's transactions
        # Already sorted by time from Step 3
        features = group[feature_cols].values

        # Get the labels for this card's transactions
        card_labels = labels.loc[group.index].values

        # Loop through each transaction on this card
        for i in range(len(group)):
            # Grab the last seq_len transactions up to and including current
            start = max(0, i - seq_len + 1)
            seq = features[start:i+1]

            # If this card doesn't have enough history yet, pad with zeros
            # E.g. card's 1st transaction: seq = [T1] → pad to [0,0,0,0,T1]
            # E.g. card's 3rd transaction: seq = [T1,T2,T3] → pad to [0,0,T1,T2,T3]
            if len(seq) < seq_len:
                padding = np.zeros((seq_len - len(seq), len(feature_cols)))
                seq = np.vstack([padding, seq])

            # Save the sequence (for LSTM)
            sequences.append(seq)

            # Save the current transaction features (for Random Forest)
            static_features.append(features[i])

            # Save the label
            targets.append(card_labels[i])

    return np.array(sequences), np.array(static_features), np.array(targets)

# ----------------------------------------------------------
# Step 6: Build the actual sequences
# This loops through all 1.29M training transactions
# grouped by 983 cards. Takes 3-5 minutes.
# ----------------------------------------------------------
print("Building training sequences...")
X_train_seq, X_train_static, y_train_seq = build_sequences(X_train, y_train, feature_cols, SEQ_LEN)
print(f"Train sequences: {X_train_seq.shape}")    # Expected: (1296675, 5, 14)
print(f"Train static: {X_train_static.shape}")      # Expected: (1296675, 14)
print(f"Train labels: {y_train_seq.shape} | Fraud: {int(y_train_seq.sum())}")

print("\nBuilding test sequences...")
X_test_seq, X_test_static, y_test_seq = build_sequences(X_test, y_test, feature_cols, SEQ_LEN)
print(f"Test sequences: {X_test_seq.shape}")        # Expected: (555719, 5, 14)
print(f"Test static: {X_test_static.shape}")          # Expected: (555719, 14)
print(f"Test labels: {y_test_seq.shape} | Fraud: {int(y_test_seq.sum())}")

Features for model: ['amt', 'city_pop', 'hour', 'month', 'distance_cardholder_merchant', 'age', 'is_weekend', 'is_night', 'velocity_1h', 'velocity_24h', 'amount_velocity_1h', 'category_encoded', 'gender_encoded', 'day_of_week_encoded']
Number of features: 14


Building training sequences...


Train sequences: (1296675, 5, 14)
Train static: (1296675, 14)
Train labels: (1296675,) | Fraud: 7506

Building test sequences...


Test sequences: (555719, 5, 14)
Test static: (555719, 14)
Test labels: (555719,) | Fraud: 2145


In [5]:
# CELL 5: LOAD PRE-TRAINED LSTM AND EXTRACT FEATURES
import zipfile, tempfile, os, joblib
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense
from tensorflow.keras.models import Model
import numpy as np

print('TensorFlow version:', tf.__version__)

# Rebuild architecture (5 timesteps, 14 features)
inp = Input(shape=(5, 14), name='sequence_input')
x = LSTM(64, return_sequences=False)(inp)
x = Dropout(0.3)(x)
x = Dense(32, activation='relu')(x)
feat = Dense(16, activation='relu', name='lstm_features')(x)
out = Dense(1, activation='sigmoid')(feat)
train_model = Model(inputs=inp, outputs=out)

# Load saved weights from .keras archive
keras_path = os.path.join('..', 'models', 'saved', '02_comparator', 'lstm_rf_keras.keras')
with zipfile.ZipFile(keras_path, 'r') as z:
    z.extract('model.weights.h5', path=tempfile.gettempdir())
weights_path = os.path.join(tempfile.gettempdir(), 'model.weights.h5')
train_model.load_weights(weights_path)
print('Pre-trained LSTM weights loaded successfully')
train_model.summary()

# Feature-extraction sub-model
lstm_model = Model(inputs=inp, outputs=feat)
print('\nLSTM feature extractor output shape:', lstm_model.output_shape)

# Extract features from sequences
print('\nExtracting LSTM features from training data...')
train_lstm_features = lstm_model.predict(X_train_seq, batch_size=512, verbose=1)
print(f'LSTM train features: {train_lstm_features.shape}')

print('\nExtracting LSTM features from test data...')
test_lstm_features = lstm_model.predict(X_test_seq, batch_size=512, verbose=1)
print(f'LSTM test features: {test_lstm_features.shape}')

# Merge LSTM features + static features
X_train_merged = np.hstack([train_lstm_features, X_train_static])
X_test_merged = np.hstack([test_lstm_features, X_test_static])
print(f'\nMerged train: {X_train_merged.shape}')
print(f'Merged test: {X_test_merged.shape}')


TensorFlow version: 2.21.0


Pre-trained LSTM weights loaded successfully


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequence_input (InputLayer)     │ (None, 5, 14)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        20,224 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_features (Dense)           │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,849 (89.25 KB)

...

1051/1086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1063/1086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1075/1086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1086/1086 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step


LSTM test features: (555719, 16)

Merged train: (1296675, 30)
Merged test: (555719, 30)


In [6]:
# CELL 6: TRAIN RF ON LSTM FEATURES + EVALUATE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, f1_score, roc_auc_score,
                             precision_score, recall_score, accuracy_score,
                             average_precision_score)

# Get LSTM fraud probabilities
print('Getting LSTM fraud probabilities...')
train_lstm_prob = train_model.predict(X_train_seq, batch_size=512, verbose=0).flatten()
test_lstm_prob = train_model.predict(X_test_seq, batch_size=512, verbose=0).flatten()

# Build feature matrix: LSTM probability + 14 static features = 15 features
X_train_final = np.column_stack([train_lstm_prob, X_train_static])
X_test_final = np.column_stack([test_lstm_prob, X_test_static])
print(f'Feature matrix: {X_train_final.shape[1]} features')

# Train RF with class weights (fast - no hyperparameter search)
n_normal = (y_train_seq == 0).sum()
n_fraud = (y_train_seq == 1).sum()
scale_pos = n_normal / n_fraud

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    class_weight={0: 1, 1: scale_pos},
    random_state=42, n_jobs=-1
)
print(f'Training RF (class_weight ratio: {scale_pos:.1f})...')
rf_model.fit(X_train_final, y_train_seq)

# Predict
hybrid_pred = rf_model.predict(X_test_final)
hybrid_proba = rf_model.predict_proba(X_test_final)[:, 1]

print()
print("="*50)
print("HYBRID MODEL: LSTM + Random Forest")
print("="*50)
print(f"Accuracy:  {accuracy_score(y_test_seq, hybrid_pred):.4f}")
print(f"Precision: {precision_score(y_test_seq, hybrid_pred):.4f}")
print(f"Recall:    {recall_score(y_test_seq, hybrid_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test_seq, hybrid_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_seq, hybrid_proba):.4f}")
print(f"PR-AUC:    {average_precision_score(y_test_seq, hybrid_proba):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test_seq, hybrid_pred, target_names=["Legit", "Fraud"]))


Getting LSTM fraud probabilities...


Feature matrix: 15 features
Training RF (class_weight ratio: 171.8)...



HYBRID MODEL: LSTM + Random Forest
Accuracy:  0.9978
Precision: 0.6522
Recall:    0.9487
F1 Score:  0.7730


ROC-AUC:   0.9980
PR-AUC:    0.9495

Classification Report:


              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    553574
       Fraud       0.65      0.95      0.77      2145

    accuracy                           1.00    555719
   macro avg       0.83      0.97      0.89    555719
weighted avg       1.00      1.00      1.00    555719



In [7]:
# CELL 7: SMOTE VARIANT
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier

smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_final, y_train_seq)
print(f'Before SMOTE: {dict(zip(*np.unique(y_train_seq, return_counts=True)))}')
print(f'After SMOTE:  {dict(zip(*np.unique(y_train_smote, return_counts=True)))}')

rf_smote = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
print('Training RF + SMOTE...')
rf_smote.fit(X_train_smote, y_train_smote)

hybrid_smote_pred = rf_smote.predict(X_test_final)
hybrid_smote_proba = rf_smote.predict_proba(X_test_final)[:, 1]

print()
print('=' * 50)
print('HYBRID MODEL: LSTM + RF + SMOTE')
print('=' * 50)
print(f'Accuracy:  {accuracy_score(y_test_seq, hybrid_smote_pred):.4f}')
print(f'Precision: {precision_score(y_test_seq, hybrid_smote_pred):.4f}')
print(f'Recall:    {recall_score(y_test_seq, hybrid_smote_pred):.4f}')
print(f'F1 Score:  {f1_score(y_test_seq, hybrid_smote_pred):.4f}')
print(f'ROC-AUC:   {roc_auc_score(y_test_seq, hybrid_smote_proba):.4f}')
print()
print('Classification Report:')
print(classification_report(y_test_seq, hybrid_smote_pred, target_names=['Legit', 'Fraud']))


Before SMOTE: {np.int64(0): np.int64(1289169), np.int64(1): np.int64(7506)}
After SMOTE:  {np.int64(0): np.int64(1289169), np.int64(1): np.int64(1289169)}
Training RF + SMOTE...



HYBRID MODEL: LSTM + RF + SMOTE
Accuracy:  0.9953
Precision: 0.4499
Recall:    0.9650
F1 Score:  0.6137


ROC-AUC:   0.9968

Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    553574
       Fraud       0.45      0.97      0.61      2145

    accuracy                           1.00    555719
   macro avg       0.72      0.98      0.81    555719
weighted avg       1.00      1.00      1.00    555719



In [8]:
# CELL 8: MODEL PARAMETERS
print('Saved RF model parameters:')
for k, v in rf_model.get_params().items():
    if k in ['n_estimators', 'max_depth', 'class_weight', 'min_samples_split', 'min_samples_leaf']:
        print(f'  {k}: {v}')
print(f'Final Hybrid F1 Score: {f1_score(y_test_seq, hybrid_pred):.4f}')


Saved RF model parameters:
  class_weight: {0: 1, 1: np.float64(171.75179856115108)}
  max_depth: 12
  min_samples_leaf: 1
  min_samples_split: 2
  n_estimators: 200
Final Hybrid F1 Score: 0.7730


In [9]:
# CELL 9: MODEL COMPARISON SUMMARY
print('=' * 65)
print('HYBRID MODEL VARIANTS COMPARISON')
print('=' * 65)

f1_saved = f1_score(y_test_seq, hybrid_pred)
p_saved = precision_score(y_test_seq, hybrid_pred)
r_saved = recall_score(y_test_seq, hybrid_pred)
auc_saved = roc_auc_score(y_test_seq, hybrid_proba)

f1_sm = f1_score(y_test_seq, hybrid_smote_pred)
p_sm = precision_score(y_test_seq, hybrid_smote_pred)
r_sm = recall_score(y_test_seq, hybrid_smote_pred)
auc_sm = roc_auc_score(y_test_seq, hybrid_smote_proba)

print(f'{"Variant":<30} {"F1":>8} {"Precision":>10} {"Recall":>8} {"ROC-AUC":>9}')
print('-' * 65)
print(f'{"LSTM + RF (class-weighted)":<30} {f1_saved:>8.4f} {p_saved:>10.4f} {r_saved:>8.4f} {auc_saved:>9.4f}')
print(f'{"LSTM + RF + SMOTE":<30} {f1_sm:>8.4f} {p_sm:>10.4f} {r_sm:>8.4f} {auc_sm:>9.4f}')
print('-' * 65)
print()
print('The class-weighted RF is the final selected hybrid variant.')


HYBRID MODEL VARIANTS COMPARISON


Variant                              F1  Precision   Recall   ROC-AUC
-----------------------------------------------------------------
LSTM + RF (class-weighted)       0.7730     0.6522   0.9487    0.9980
LSTM + RF + SMOTE                0.6137     0.4499   0.9650    0.9968
-----------------------------------------------------------------

The class-weighted RF is the final selected hybrid variant.


In [10]:
# Instead of 16 LSTM features, use LSTM's fraud prediction as 1 feature
train_lstm_prob = train_model.predict(X_train_seq, batch_size=512, verbose=1).flatten()
test_lstm_prob = train_model.predict(X_test_seq, batch_size=512, verbose=1).flatten()

# Merge: 1 LSTM probability + 14 static features = 15 features
X_train_slim = np.column_stack([train_lstm_prob, X_train_static])
X_test_slim = np.column_stack([test_lstm_prob, X_test_static])

print(f"Slim train: {X_train_slim.shape}")
print(f"Slim test: {X_test_slim.shape}")

# Train RF on slim features
rf_slim = RandomForestClassifier(
    n_estimators=200, max_depth=12,
    class_weight={0:1, 1:150},
    random_state=42, n_jobs=-1
)
rf_slim.fit(X_train_slim, y_train_seq)
slim_pred = rf_slim.predict(X_test_slim)
slim_proba = rf_slim.predict_proba(X_test_slim)[:, 1]

print("\n" + "="*50)
print("HYBRID v2: LSTM probability + RF")
print("="*50)
print(f"Accuracy:  {accuracy_score(y_test_seq, slim_pred):.4f}")
print(f"Precision: {precision_score(y_test_seq, slim_pred):.4f}")
print(f"Recall:    {recall_score(y_test_seq, slim_pred):.4f}")
print(f"F1 Score:  {f1_score(y_test_seq, slim_pred):.4f}")
print(f"ROC-AUC:   {roc_auc_score(y_test_seq, slim_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test_seq, slim_pred, target_names=['Legit', 'Fraud']))

   1/2533 ━━━━━━━━━━━━━━━━━━━━ 2:09 51ms/step

  11/2533 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step  

  23/2533 ━━━━━━━━━━━━━━━━━━━━ 12s 5ms/step

  35/2533 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step

  47/2533 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step

...

1084/1086 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step

1086/1086 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step


Slim train: (1296675, 15)
Slim test: (555719, 15)



HYBRID v2: LSTM probability + RF
Accuracy:  0.9980
Precision: 0.6685
Recall:    0.9469
F1 Score:  0.7837


ROC-AUC:   0.9981

Classification Report:
              precision    recall  f1-score   support

       Legit       1.00      1.00      1.00    553574
       Fraud       0.67      0.95      0.78      2145

    accuracy                           1.00    555719
   macro avg       0.83      0.97      0.89    555719
weighted avg       1.00      1.00      1.00    555719



## Final result summary — Hybrid 1 (LSTM + RF)

| Pipeline | F1 | Precision | Recall | ROC-AUC | Source |
|---|---:|---:|---:|---:|---|
| **Corrected (composite-key merge)** | **0.7892** | 0.6770 | 0.9459 | 0.9981 | `verified_metrics.json` → `LSTM_reproduced_baseline` |
| Historical / broken (positional `.values` attach) | 0.4747 | — | — | — | `FYP_Hybrid_Model_BROKEN.ipynb` (audit-preserved only) |

The 0.4747 figure is **not** the current Hybrid 1 result and should not be cited as such in the dissertation. It is preserved purely for transparency about the methodology fix.

Re-run the corrected pipeline end-to-end from the CLI with:

```
python run_gap_experiments.py
```

That script uses the same `_with_ids.csv` files and seed=42 and writes the authoritative metrics block to `verified_metrics.json`.


# Final 3-Model Staged Study Summary

This notebook implements the **Hybrid Comparator** of the staged 3-model study. Its role is to test sequence/temporal modelling against the tabular Baseline and Proposed Model. The headline 3-model comparison (and the Proposed Model component analysis) is also recorded in `04_BDS_GA.ipynb`.

## Main Model Comparison

| Category | Model | F1 | Precision | Recall | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|---|
| Baseline | XGBoost (SMOTE+tuned) | 0.8646 | 0.9297 | 0.8079 | 0.9972 | 0.9092 |
| Hybrid Comparator | LSTM + Random Forest | 0.7892 | 0.6770 | 0.9459 | 0.9981 | n/a (not recorded) |
| Proposed Model | AE + BDS(GA) + XGBoost | 0.8706 | 0.9338 | 0.8154 | 0.9976 | 0.9158 |

## Proposed Model Component Analysis

| Variant | What It Tests | F1 | Precision | Recall | ROC-AUC | PR-AUC |
|---|---|---|---|---|---|---|
| XGBoost only | Without anomaly/BDS | 0.8646 | 0.9297 | 0.8079 | 0.9972 | 0.9092 |
| AE + XGBoost | Adds autoencoder | 0.8690 | 0.9369 | 0.8103 | 0.9973 | 0.9142 |
| AE + BDS(GA) + XGBoost | Adds behavioural scores | 0.8706 | 0.9338 | 0.8154 | 0.9976 | 0.9158 |

LSTM result here reflects post row-alignment fix; the historical 0.4747 figure is audit-preserved in `notebooks/archive/FYP_Hybrid_Model_BROKEN.ipynb` and is not for citation. All metrics at threshold = 0.5, sourced from `results/verified_metrics.json`.